# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# I chose K-Means clustering because the goal here is to find groups
# of content with similar performance patterns rather than predict a label.

method = "K-Means clustering"

print("Method:", method)
print(
    "I chose clustering because this lane is about discovering "
    "content performance archetypes. I will use the available numeric "
    "content and performance signals, scale them before clustering, "
    "and compare candidate cluster counts using silhouette score."
)

Method: K-Means clustering
I chose clustering because this lane is about discovering content performance archetypes. I will use the available numeric content and performance signals, scale them before clustering, and compare candidate cluster counts using silhouette score.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# This is an unsupervised clustering task, so there is no target
# variable that needs a train/test split.

print(
    "No traditional train/test split is used because clustering is unsupervised. "
    "I will use the full eligible dataset for the clustering step and check "
    "whether the grouping is stable and reasonably separated."
)

No traditional train/test split is used because clustering is unsupervised. I will use the full eligible dataset for the clustering step and check whether the grouping is stable and reasonably separated.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load the feature table used for this lane.
# Change this path only if your actual file has a different location.
df = pd.read_csv("data/processed/refresh_feature_vector.csv")

features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
]

features = [c for c in features if c in df.columns]

X = df[features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Let the data choose the number of clusters.
scores = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    scores.append((k, score))

scores = pd.DataFrame(scores, columns=["k", "silhouette_score"])

best_k = int(
    scores.loc[scores["silhouette_score"].idxmax(), "k"]
)

final_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = final_model.fit_predict(X_scaled)

print("Selected clusters:", best_k)
print("Best silhouette score:", round(scores["silhouette_score"].max(), 3))

cluster_profile = (
    df.groupby("cluster")[features]
    .median()
    .round(2)
)

cluster_profile

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/refresh_feature_vector.csv'

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
print(
    "The clusters should be treated as performance patterns rather than "
    "true labels. A cluster can summarize pages with similar observed "
    "signals, but it does not prove that the pages have the same underlying "
    "cause or content quality."
)

print(
    "The main limitations are sensitivity to the selected features and "
    "scaling, and the fact that the clusters can change if the underlying "
    "content mix changes. I would therefore use the clusters as a practical "
    "segmentation tool and keep human review for the final interpretation."
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.